# **Vision-Language Models (VLMs)**

A **Vision-Language Model** jointly understands images and text. Depending on the design, a VLM can match an image to a caption, describe an image in words, or answer free-form questions about it (VQA). The unifying idea is to encode pixels and tokens into representations that can interact.

Two broad architecture families dominate:

1. **Dual-encoder (contrastive)** — e.g. **CLIP**, **SigLIP**. Separate image and text encoders projected into a shared embedding space; great for retrieval and zero-shot classification, but cannot *generate* text.
2. **Fusion / generative** — e.g. **BLIP / BLIP-2**, **LLaVA**, **Flamingo**. A vision encoder is connected to a language model so the model can *produce* text conditioned on the image (captioning, VQA, chat).

## **1. Dual-encoder VLMs (CLIP-style)**

A dual encoder runs an image encoder $f_\text{img}$ (ViT or CNN) and a text encoder $f_\text{txt}$ (Transformer) independently, then aligns their outputs in a shared space with a contrastive loss (see the *cross-modal retrieval* notebook). Key properties:

- **No cross-modal attention** during encoding — the two towers never see each other until the final similarity. This makes embeddings **cacheable** and retrieval cheap (precompute all image vectors once).
- Enables **zero-shot classification**: embed the image, embed candidate label prompts ("a photo of a {class}"), pick the most similar text.
- Cannot caption or answer questions — there is no decoder.

## **2. Fusion / generative VLMs**

To *generate* language about an image, the visual features must condition a language decoder. Representative designs:

- **BLIP-2** — freezes a pretrained image encoder *and* a pretrained LLM, and trains a lightweight **Q-Former** (querying transformer) in between. A set of learned query tokens cross-attend to image features and produce a compact visual prefix the LLM consumes. Very parameter-efficient.
- **LLaVA** — takes CLIP ViT image features and maps them through a simple MLP **projection** into the LLM's token embedding space; the projected "visual tokens" are prepended to the text prompt and the LLM is instruction-tuned. Conceptually the simplest connector.
- **Flamingo** — inserts **gated cross-attention** layers into a frozen LLM so text tokens attend to image features; supports interleaved image-text and few-shot prompting.

The shared theme: a **connector** (Q-Former, MLP, or cross-attention) bridges a vision encoder to an LLM, often keeping both backbones frozen.

## **3. Image-text alignment, captioning, and VQA**

- **Alignment** is the foundation: contrastive pretraining (CLIP) or image-text matching heads (BLIP) teach the model which images and texts correspond.
- **Captioning** = conditional language generation: $P(\text{text} \mid \text{image})$, decoded autoregressively from visual features.
- **VQA** = generation conditioned on *both* image and a question: $P(\text{answer} \mid \text{image}, \text{question})$. Modern open-ended VQA is just captioning with the question prepended to the prompt.

## **4. HF usage: zero-shot classification with CLIP**

```python
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import torch

model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

image = Image.open("cat.jpg")
labels = ["a photo of a cat", "a photo of a dog", "a photo of a car"]

inputs = processor(text=labels, images=image, return_tensors="pt", padding=True)
with torch.no_grad():
    outputs = model(**inputs)

# logits_per_image: (num_images, num_texts) image-text similarity scores
probs = outputs.logits_per_image.softmax(dim=-1)
print({lbl: float(p) for lbl, p in zip(labels, probs[0])})
```

## **5. HF usage: image captioning with BLIP**

```python
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image

processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base")

image = Image.open("street.jpg").convert("RGB")
inputs = processor(images=image, return_tensors="pt")
out = model.generate(**inputs, max_new_tokens=30)
print(processor.decode(out[0], skip_special_tokens=True))
```

**References**
- Radford et al., *CLIP*, 2021.
- Li et al., *BLIP-2: Bootstrapping Language-Image Pre-training with Frozen Image Encoders and LLMs*, 2023.
- Liu et al., *Visual Instruction Tuning* (LLaVA), 2023.
- Alayrac et al., *Flamingo*, 2022.